In [ ]:
%reload_ext autoreload
%autoreload 2

# Dataset Description
The shared folder contains a bunch of images with extension .tif and each of these images contain corresponding json file with same filename which contains the polygon annotations. The polygon annotations indicate each trees that are annotated using 'labelme' tool: https://github.com/wkentaro/labelme.

The code below will demonstrate:
1) How to read the tif file?
2) How to read the json file with polygon annoatations?
3) How to overlay the polygon annotations on top of image for visualization?

You will need to figure out how to use these annotations for model training yourself. But I have also shared with you how these are used in Mask R-CNN model training for same tasks in the email. You can look at the preprocessing.py file to get an idea.

In [ ]:
pip install rasterio


In [ ]:
import json
import cv2
import numpy as np
import os
import rasterio
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import Polygon
from PIL import ImageDraw, Image
from matplotlib import pyplot as plt 

In [ ]:
def polygons_to_mask(image_size, polygons):
    """
    Draw each polygon with a unique ID (1..N) on the mask.
    """
    h, w = image_size
    mask = np.zeros((h, w), dtype=np.int32)

    for idx, poly in enumerate(polygons, start=1):
        pts = np.array(poly, dtype=np.int32)
        cv2.fillPoly(mask, [pts], idx)

    return mask

In [ ]:
import pandas as pd

In [ ]:
filename = "McCormick_25Apr01_0_1"
annotation_file_path = f"/kaggle/input/tree-data-for-instance-segmentation/tree_data_for_instance_segmentation/{filename}.json"
image_file_path = f"/kaggle/input/tree-data-for-instance-segmentation/tree_data_for_instance_segmentation/{filename}.tif"


# Read RGB image
with rasterio.open(image_file_path) as src:
    image = np.moveaxis(src.read(), 0, -1)  # (H, W, C)

# Read annotation JSON
annotation_df = pd.read_json(annotation_file_path, orient='index').transpose()

# Collect all polygon coordinates
xy_coords = []
for shapes in annotation_df['shapes']:
    for shape in shapes:
        points = shape["points"]
        xy_coords.append([tuple(p) for p in points])

# --- Create instance mask ---
mask = polygons_to_mask(image.shape[:2], xy_coords)

# --- Assign random color to each instance ---
unique_ids = np.unique(mask)
unique_ids = unique_ids[unique_ids != 0]  # skip background

np.random.seed(42)
colors = {uid: np.random.randint(0, 255, size=3).tolist() for uid in unique_ids}

# --- Create RGB overlay ---
overlay = np.zeros_like(image, dtype=np.uint8)
for uid in unique_ids:
    overlay[mask == uid] = colors[uid]

# --- Blend overlay with image ---
blended = cv2.addWeighted(image, 0.85, overlay, 0.35, 0).astype('uint8')


In [ ]:
plt.imshow(mask)

In [ ]:
plt.imshow(blended)

In [ ]:
import os
import json
import math
import cv2
import numpy as np
import rasterio
from rasterio.windows import Window
from PIL import Image
from shapely.geometry import Polygon
from shapely.errors import TopologicalError
from shapely.validation import explain_validity

##### UPDATE START #####
''' image_files: Provide the list of tif files below that you will be using for training and validation
    If the file is already used in previous iteration, that's fine since the TRACKING.json will store that information
    and it will be checked before running the code

    image_folder: path to the folder containing training images

    annotation_folder: path to the folder containing annotation json files from labelme. NOTE that the json filename should exactly
    match the tif image filename. For example: if tif filename is McCormick_25Apr01_0_3.tif then json filename should be
    McCormick_25Apr01_0_3.json

    preprocessed_folder: path to the folder to save preprocessed files

    tracking_json_path: path to the json file to keep track of images already used for training
'''
##### UPDATE END #####

# ---- CONFIG ----
image_folder = "/kaggle/input/tree-data-for-instance-segmentation/tree_data_for_instance_segmentation/"
annotation_folder = "/kaggle/input/tree-data-for-instance-segmentation/tree_data_for_instance_segmentation/"
preprocessed_folder = "/kaggle/working/preprocessed/"
tracking_json_path = "/kaggle/working/TRACKING.json"

# Automatically get all .tif files
image_files = [f for f in os.listdir(image_folder) if f.endswith('.tif')]

# If you intentionally want to skip index 1, keep this line; otherwise remove it.
# del image_files[1]

PATCH_SIZE = 1024
STRIDE = PATCH_SIZE // 2
KEEP_THRESHOLD = 0.50  # keep object if ≥50% area inside patch
RESIZE_FACTOR = 0.60
# -----------------

# Load or initialize tracking
try:
    with open(tracking_json_path, 'r') as fp:
        tracking = json.load(fp)
    preprocessed_files = tracking.get("preprocessed", [])
    # set next start id to last saved + 1 to avoid id collisions
    latest_start_id = int(tracking.get("latest_start_id", 1000)) + 1
except Exception:
    tracking = {"preprocessed": [], "latest_start_id": 1000,
                "mapping": {"id_to_filename": [], "filename_to_id": []}}
    preprocessed_files = []
    latest_start_id = 1000

# Output folders
out_dir_png  = os.path.join(preprocessed_folder, "train_test_PNG")
out_dir_bbox = os.path.join(preprocessed_folder, "train_test_BBOX")
out_dir_mask = os.path.join(preprocessed_folder, "train_test_MASK")
os.makedirs(out_dir_png, exist_ok=True)
os.makedirs(out_dir_bbox, exist_ok=True)
os.makedirs(out_dir_mask, exist_ok=True)

def create_polygon_mask(shape, polygon_points):
    mask = np.zeros(shape, dtype=np.uint8)
    cv2.fillPoly(mask, [np.array(polygon_points, np.int32)], 1)
    return mask

# Helper: safe Polygon creation that attempts to fix invalid polygons via buffer(0)
def make_valid_polygon(pts, image_file=None):
    poly = Polygon(pts)
    if not poly.is_valid or poly.area == 0:
        try:
            fixed = poly.buffer(0)
            if fixed.is_valid and fixed.area > 0:
                return fixed
            else:
                # still invalid
                if image_file:
                    print(f"⚠️ Polygon invalid after buffer(0) for {image_file}: {explain_validity(poly)}")
                return None
        except Exception as e:
            if image_file:
                print(f"⚠️ Exception while fixing polygon for {image_file}: {e}")
            return None
    return poly

# Main loop
for image_file in image_files:
    print(f"\nCurrently working on {image_file}")

    # if this image already processed then skip
    if image_file in preprocessed_files:
        print("------WARNING-------")
        print(f"{image_file} is already processed according to {tracking_json_path}. If you want to reprocess it, remove it from that file.")
        print("--------------------")
        continue

    # We'll update tracking for the image; patch IDs will start from this value
    patch_id = latest_start_id

    # Add to tracking (we'll update latest_start_id at the end of this image processing)
    tracking["preprocessed"].append(image_file)
    tracking["mapping"]["id_to_filename"].append({patch_id: image_file})
    tracking["mapping"]["filename_to_id"].append({image_file: patch_id})

    filename = image_file.split(".tif")[0]

    # --- Load annotations ---
    json_path = os.path.join(annotation_folder, f"{filename}.json")
    if not os.path.exists(json_path):
        print(f"⚠️ Annotation JSON not found for {image_file} at {json_path}. Skipping this image.")
        continue

    with open(json_path) as f:
        data = json.load(f)

    # --- Load image (resized) ---
    with rasterio.open(os.path.join(image_folder, image_file)) as src:
        image = np.moveaxis(src.read(), 0, -1)  # HWC
        H, W, _ = image.shape

        # Step 1: resize
        new_H, new_W = int(H * RESIZE_FACTOR), int(W * RESIZE_FACTOR)
        if new_H <= 0 or new_W <= 0:
            print(f"⚠️ Resized dimensions invalid for {image_file}: {new_H}x{new_W}. Skipping.")
            continue
        image_resized = cv2.resize(image, (new_W, new_H), interpolation=cv2.INTER_LINEAR)

        # Step 2: pad to multiples of patch size
        pad_h = (math.ceil(new_H / PATCH_SIZE) * PATCH_SIZE) - new_H
        pad_w = (math.ceil(new_W / PATCH_SIZE) * PATCH_SIZE) - new_W
        image_padded = np.pad(image_resized,
                              ((0, pad_h), (0, pad_w), (0, 0)),
                              mode="reflect")
        H_pad, W_pad, _ = image_padded.shape

        # Scale polygons (and validate)
        shapes_scaled = []
        for shape in data.get("shapes", []):
            pts = np.array(shape["points"], dtype=float)
            pts *= RESIZE_FACTOR  # rescale to match resized image
            # Ensure points are within bounds (clamp)
            pts[:, 0] = np.clip(pts[:, 0], 0, W_pad - 1)
            pts[:, 1] = np.clip(pts[:, 1], 0, H_pad - 1)
            shapes_scaled.append({**shape, "points": pts.tolist()})

        # Step 3+4: patching
        for y in range(0, H_pad - PATCH_SIZE + 1, STRIDE):
            for x in range(0, W_pad - PATCH_SIZE + 1, STRIDE):
                patch = image_padded[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

                patch_shapes = []
                for shape in shapes_scaled:
                    pts = np.array(shape["points"], dtype=float)
                    # Skip degenerate shapes (less than 3 points)
                    if pts.shape[0] < 3:
                        continue

                    # Create polygon safely
                    poly = make_valid_polygon(pts, image_file=image_file)
                    if poly is None:
                        # couldn't fix polygon, skip it
                        continue

                    # Create patch box polygon
                    patch_box = Polygon([(x, y), (x + PATCH_SIZE, y), (x + PATCH_SIZE, y + PATCH_SIZE), (x, y + PATCH_SIZE)])

                    try:
                        inter = poly.intersection(patch_box)
                    except TopologicalError as e:
                        # try to fix both geometries and retry intersection
                        try:
                            poly = poly.buffer(0)
                            patch_box = patch_box.buffer(0)
                            inter = poly.intersection(patch_box)
                        except Exception:
                            print(f"⚠️ Skipping polygon due to TopologicalError at image {image_file}, patch ({x},{y}): {e}")
                            continue
                    except Exception as e:
                        print(f"⚠️ Unexpected error during intersection at {image_file}, patch ({x},{y}): {e}")
                        continue

                    # If intersection is empty or negligible, skip
                    if inter.is_empty:
                        continue

                    # Some intersections may produce geometry collections; handle MultiPolygon or Polygon
                    polys_to_iterate = []
                    if inter.geom_type == "Polygon":
                        polys_to_iterate = [inter]
                    elif inter.geom_type == "MultiPolygon":
                        polys_to_iterate = list(inter.geoms)
                    else:
                        # Not a polygonal intersection (e.g., LINESTRING) -> skip
                        continue

                    # Only keep polygons that have reasonable area relative to original polygon
                    for sub_poly in polys_to_iterate:
                        try:
                            # If original poly area is zero (shouldn't happen after make_valid_polygon), skip
                            if poly.area == 0:
                                continue
                            # area ratio
                            area_ratio = sub_poly.area / poly.area if poly.area > 0 else 0
                        except Exception:
                            area_ratio = 0

                        if area_ratio >= KEEP_THRESHOLD:
                            # Extract exterior ring coords and shift to patch-local coords
                            inter_coords = np.array(sub_poly.exterior.coords)
                            shifted = [[float(px - x), float(py - y)] for px, py in inter_coords]
                            # Ensure coordinates are within [0, PATCH_SIZE)
                            for pt in shifted:
                                pt[0] = max(0.0, min(float(PATCH_SIZE - 1), pt[0]))
                                pt[1] = max(0.0, min(float(PATCH_SIZE - 1), pt[1]))

                            patch_shapes.append({
                                "label": shape.get("label", "tree"),
                                "points": shifted,
                                "group_id": None,
                                "shape_type": "polygon",
                                "flags": {}
                            })

                if len(patch_shapes) == 0:
                    continue

                # Save patch image
                png_name = f"{patch_id}.png"
                Image.fromarray(patch).save(os.path.join(out_dir_png, png_name))

                # Save JSON (LabelMe-like)
                new_json = {
                    "version": data.get("version", "5.8.3"),
                    "flags": {},
                    "shapes": patch_shapes,
                    "imagePath": png_name,
                    "imageHeight": PATCH_SIZE,
                    "imageWidth": PATCH_SIZE,
                }
                with open(os.path.join(out_dir_png, f"{patch_id}.json"), "w") as jf:
                    json.dump(new_json, jf, indent=2)

                # ---- Create mask PNG ----
                mask = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.uint8)
                for shp in patch_shapes:
                    pts = np.array(shp["points"], dtype=np.int32)
                    # skip shapes that are degenerate
                    if pts.size == 0:
                        continue
                    cv2.fillPoly(mask, [pts], 1)

                mask_name = f"{patch_id}_mask.png"
                mask_path = os.path.join(out_dir_mask, mask_name)
                cv2.imwrite(mask_path, (mask * 255).astype(np.uint8))  # scale 0/1 → 0/255

                # ---- Draw bounding box preview ----
                if patch_shapes:
                    preview = patch.copy()
                    if preview.dtype != np.uint8:
                        pmin, pmax = np.percentile(preview, [2, 98])
                        if pmax > pmin:
                            preview = np.clip((preview - pmin) * (255.0 / (pmax - pmin)), 0, 255).astype(np.uint8)
                        else:
                            preview = np.clip(preview, 0, 255).astype(np.uint8)

                    for shp in patch_shapes:
                        xs = [p[0] for p in shp["points"]]
                        ys = [p[1] for p in shp["points"]]
                        box_x_min, box_y_min = int(max(0, min(xs))), int(max(0, min(ys)))
                        box_x_max, box_y_max = int(min(PATCH_SIZE - 1, max(xs))), int(min(PATCH_SIZE - 1, max(ys)))

                        # ensure box has positive area
                        if box_x_max > box_x_min and box_y_max > box_y_min:
                            cv2.rectangle(preview,
                                          (box_x_min, box_y_min),
                                          (box_x_max, box_y_max),
                                          (0, 255, 0), 2)

                    bbox_png = os.path.join(out_dir_bbox, f"{patch_id}_bbox.png")
                    cv2.imwrite(bbox_png, cv2.cvtColor(preview, cv2.COLOR_RGB2BGR))

                print(f"Saved patch, JSON, and bbox preview → {png_name}, {patch_id}_bbox.png")

                patch_id += 1  # next patch id

    # After finishing this image, update latest_start_id to last used id (so next image won't collide)
    tracking["latest_start_id"] = patch_id - 1
    latest_start_id = tracking["latest_start_id"] + 1

    # Save tracking after each image to avoid losing progress
    try:
        with open(tracking_json_path, 'w') as fp:
            json.dump(tracking, fp, indent=2)
    except Exception as e:
        print(f"⚠️ Failed to write tracking file: {e}")

print("\nDone — patches created with resized & padded workflow")


In [ ]:
# import os
# import json
# import torch
# import numpy as np
# from PIL import Image
# from torch.utils.data import Dataset, DataLoader
# import cv2
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches
# import random

# class TreeDataset(Dataset):
#     def __init__(self, image_dir, mask_dir, transforms=None, augment=False):
#         """
#         Args:
#             image_dir: Directory with PNG images
#             mask_dir: Directory with mask PNG files
#             transforms: Optional transforms to apply
#             augment: Whether to apply augmentations (True for training)
#         """
#         self.image_dir = image_dir
#         self.mask_dir = mask_dir
#         self.transforms = transforms
#         self.augment = augment
        
#         # Get all image files
#         self.image_files = sorted([f for f in os.listdir(image_dir) 
#                                    if f.endswith('.png') and not f.endswith('_mask.png')])
        
#     def __len__(self):
#         return len(self.image_files)
    
#     def apply_augmentation(self, image, mask):
#         """
#         Apply simple augmentations to image and mask
#         """
#         # Horizontal flip (50% chance)
#         if random.random() > 0.5:
#             image = np.fliplr(image).copy()
#             mask = np.fliplr(mask).copy()
        
#         # Vertical flip (50% chance)
#         if random.random() > 0.5:
#             image = np.flipud(image).copy()
#             mask = np.flipud(mask).copy()
        
#         # Random brightness adjustment (±20%)
#         if random.random() > 0.5:
#             factor = random.uniform(0.8, 1.2)
#             image = np.clip(image * factor, 0, 255).astype(np.uint8)
        
#         # Random rotation (90, 180, or 270 degrees)
#         if random.random() > 0.5:
#             k = random.choice([1, 2, 3])  # 90, 180, or 270 degrees
#             image = np.rot90(image, k).copy()
#             mask = np.rot90(mask, k).copy()
        
#         return image, mask
    
#     def extract_instances_from_mask(self, mask, min_area=50):
#         """
#         Extract individual instances from binary mask using connected components
#         Returns list of instance masks and their bounding boxes
#         """
#         mask_uint8 = (mask > 0).astype(np.uint8) * 255
        
#         # Find connected components
#         num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
#             mask_uint8, connectivity=8
#         )
        
#         instance_masks = []
#         boxes = []
        
#         # Skip background (label 0)
#         for label in range(1, num_labels):
#             area = stats[label, cv2.CC_STAT_AREA]
            
#             if area < min_area:
#                 continue
            
#             # Extract bounding box
#             x = stats[label, cv2.CC_STAT_LEFT]
#             y = stats[label, cv2.CC_STAT_TOP]
#             w = stats[label, cv2.CC_STAT_WIDTH]
#             h = stats[label, cv2.CC_STAT_HEIGHT]
            
#             # Store as [x1, y1, x2, y2]
#             boxes.append([float(x), float(y), float(x + w), float(y + h)])
            
#             # Extract instance mask
#             instance_mask = (labels == label).astype(np.uint8)
#             instance_masks.append(instance_mask)
        
#         return instance_masks, boxes
    
#     def __getitem__(self, idx):
#         # Load image
#         img_name = self.image_files[idx]
#         img_id = img_name.split('.')[0]
#         img_path = os.path.join(self.image_dir, img_name)
#         image = Image.open(img_path).convert('RGB')
        
#         # Load mask
#         mask_name = f"{img_id}_mask.png"
#         mask_path = os.path.join(self.mask_dir, mask_name)
#         mask = Image.open(mask_path).convert('L')
        
#         # Convert to numpy
#         image = np.array(image)
#         mask = np.array(mask)
        
#         # Apply augmentation if enabled
#         if self.augment:
#             image, mask = self.apply_augmentation(image, mask)
        
#         # Extract individual instances from the mask
#         instance_masks, boxes = self.extract_instances_from_mask(mask)
        
#         # If no instances found, create dummy data
#         if len(boxes) == 0:
#             boxes = torch.zeros((0, 4), dtype=torch.float32)
#             labels = torch.zeros((0,), dtype=torch.int64)
#             masks = torch.zeros((0, mask.shape[0], mask.shape[1]), dtype=torch.uint8)
#         else:
#             boxes = torch.as_tensor(boxes, dtype=torch.float32)
#             labels = torch.ones((len(boxes),), dtype=torch.int64)
#             # Stack instance masks
#             masks = torch.as_tensor(np.stack(instance_masks), dtype=torch.uint8)
        
#         target = {
#             'boxes': boxes,
#             'labels': labels,
#             'masks': masks,
#             'image_id': torch.tensor([idx])
#         }
        
#         # Convert image to tensor
#         image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        
#         if self.transforms:
#             image = self.transforms(image)
        
#         return image, target


# def get_dataloaders(preprocessed_folder, batch_size=2, train_split=0.9):
#     """
#     Create train and validation dataloaders
#     """
#     png_dir = os.path.join(preprocessed_folder, "train_test_PNG")
#     mask_dir = os.path.join(preprocessed_folder, "train_test_MASK")
    
#     # Create datasets with augmentation for training only
#     train_dataset_full = TreeDataset(png_dir, mask_dir, augment=False)
#     val_dataset_full = TreeDataset(png_dir, mask_dir, augment=False)
    
#     # Split into train and validation
#     dataset_size = len(train_dataset_full)
#     train_size = int(train_split * dataset_size)
#     val_size = dataset_size - train_size
    
#     # Get indices for split
#     indices = list(range(dataset_size))
#     train_indices = indices[:train_size]
#     val_indices = indices[train_size:]
    
#     # Create subset datasets
#     train_dataset = torch.utils.data.Subset(train_dataset_full, train_indices)
#     val_dataset = torch.utils.data.Subset(val_dataset_full, val_indices)
    
#     # Create dataloaders
#     train_loader = DataLoader(
#         train_dataset,
#         batch_size=batch_size,
#         shuffle=True,
#         num_workers=2,
#         collate_fn=lambda x: tuple(zip(*x))
#     )
    
#     val_loader = DataLoader(
#         val_dataset,
#         batch_size=batch_size,
#         shuffle=False,
#         num_workers=2,
#         collate_fn=lambda x: tuple(zip(*x))
#     )
    
#     return train_loader, val_loader


# def visualize_samples(dataloader, num_samples=2):
#     """
#     Visualize samples from the dataloader
#     """
#     # Get a batch
#     images, targets = next(iter(dataloader))
    
#     fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
#     if num_samples == 1:
#         axes = axes.reshape(1, -1)
    
#     for i in range(min(num_samples, len(images))):
#         img = images[i].permute(1, 2, 0).numpy()
#         target = targets[i]
        
#         # Original image
#         axes[i, 0].imshow(img)
#         axes[i, 0].set_title(f"Image {i+1}")
#         axes[i, 0].axis('off')
        
#         # Image with bounding boxes
#         axes[i, 1].imshow(img)
#         for box in target['boxes']:
#             x1, y1, x2, y2 = box
#             rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1, 
#                                      linewidth=2, edgecolor='r', facecolor='none')
#             axes[i, 1].add_patch(rect)
#         axes[i, 1].set_title(f"Image {i+1} with Boxes (n={len(target['boxes'])})")
#         axes[i, 1].axis('off')
        
#         # Show all instance masks combined
#         if len(target['masks']) > 0:
#             combined_mask = target['masks'].sum(dim=0).numpy()
#         else:
#             combined_mask = np.zeros((img.shape[0], img.shape[1]))
#         axes[i, 2].imshow(combined_mask, cmap='gray')
#         axes[i, 2].set_title(f"Instances: {len(target['masks'])}")
#         axes[i, 2].axis('off')
    
#     plt.tight_layout()
#     plt.show()

In [ ]:
import os
import json
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random

class TreeDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transforms=None, augment=False):
        """
        Args:
            image_dir: Directory with PNG images
            mask_dir: Directory with mask PNG files
            transforms: Optional transforms to apply
            augment: Whether to apply augmentations (True for training)
        """
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transforms = transforms
        self.augment = augment
        
        # Get all image files
        self.image_files = sorted([f for f in os.listdir(image_dir) 
                                   if f.endswith('.png') and not f.endswith('_mask.png')])
        
    def __len__(self):
        return len(self.image_files)
    
    def apply_augmentation(self, image, mask):
        """
        Apply simple augmentations to image and mask
        """
        # Horizontal flip (50% chance)
        if random.random() > 0.5:
            image = np.fliplr(image).copy()
            mask = np.fliplr(mask).copy()
        
        # Vertical flip (50% chance)
        if random.random() > 0.5:
            image = np.flipud(image).copy()
            mask = np.flipud(mask).copy()
        
        # Random brightness adjustment (±20%)
        if random.random() > 0.5:
            factor = random.uniform(0.8, 1.2)
            image = np.clip(image * factor, 0, 255).astype(np.uint8)
        
        # Random rotation (90, 180, or 270 degrees)
        if random.random() > 0.5:
            k = random.choice([1, 2, 3])  # 90, 180, or 270 degrees
            image = np.rot90(image, k).copy()
            mask = np.rot90(mask, k).copy()
        
        return image, mask
    
    def extract_instances_from_mask(self, mask, min_area=50):
        """
        Extract individual instances from binary mask using connected components
        Returns list of instance masks and their bounding boxes
        """
        mask_uint8 = (mask > 0).astype(np.uint8) * 255
        
        # Find connected components
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
            mask_uint8, connectivity=8
        )
        
        instance_masks = []
        boxes = []
        
        # Skip background (label 0)
        for label in range(1, num_labels):
            area = stats[label, cv2.CC_STAT_AREA]
            
            if area < min_area:
                continue
            
            # Extract bounding box
            x = stats[label, cv2.CC_STAT_LEFT]
            y = stats[label, cv2.CC_STAT_TOP]
            w = stats[label, cv2.CC_STAT_WIDTH]
            h = stats[label, cv2.CC_STAT_HEIGHT]
            
            # Store as [x1, y1, x2, y2]
            boxes.append([float(x), float(y), float(x + w), float(y + h)])
            
            # Extract instance mask
            instance_mask = (labels == label).astype(np.uint8)
            instance_masks.append(instance_mask)
        
        return instance_masks, boxes
    
    def __getitem__(self, idx):
        # Load image
        img_name = self.image_files[idx]
        img_id = img_name.split('.')[0]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        
        # Load mask
        mask_name = f"{img_id}_mask.png"
        mask_path = os.path.join(self.mask_dir, mask_name)
        mask = Image.open(mask_path).convert('L')
        
        # Convert to numpy
        image = np.array(image)
        mask = np.array(mask)
        
        # Apply augmentation if enabled
        if self.augment:
            image, mask = self.apply_augmentation(image, mask)
        
        # Extract individual instances from the mask
        instance_masks, boxes = self.extract_instances_from_mask(mask)
        
        # If no instances found, create dummy data
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            masks = torch.zeros((0, mask.shape[0], mask.shape[1]), dtype=torch.uint8)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.ones((len(boxes),), dtype=torch.int64)
            # Stack instance masks
            masks = torch.as_tensor(np.stack(instance_masks), dtype=torch.uint8)
        
        target = {
            'boxes': boxes,
            'labels': labels,
            'masks': masks,
            'image_id': torch.tensor([idx])
        }
        
        # Convert image to tensor
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        
        if self.transforms:
            image = self.transforms(image)
        
        return image, target


def get_dataloaders(preprocessed_folder, batch_size=2, train_split=0.9):
    """
    Create train and validation dataloaders
    """
    png_dir = os.path.join(preprocessed_folder, "train_test_PNG")
    mask_dir = os.path.join(preprocessed_folder, "train_test_MASK")
    
    # Create datasets with augmentation for training only
    train_dataset_full = TreeDataset(png_dir, mask_dir, augment=True)
    val_dataset_full = TreeDataset(png_dir, mask_dir, augment=False)
    
    # Split into train and validation
    dataset_size = len(train_dataset_full)
    train_size = int(train_split * dataset_size)
    val_size = dataset_size - train_size
    
    # Get indices for split
    indices = list(range(dataset_size))
    train_indices = indices[:train_size]
    val_indices = indices[train_size:]
    
    # Create subset datasets
    train_dataset = torch.utils.data.Subset(train_dataset_full, train_indices)
    val_dataset = torch.utils.data.Subset(val_dataset_full, val_indices)
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        collate_fn=lambda x: tuple(zip(*x))
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        collate_fn=lambda x: tuple(zip(*x))
    )
    
    return train_loader, val_loader


def visualize_samples(dataloader, num_samples=2):
    """
    Visualize samples from the dataloader
    """
    # Get a batch
    images, targets = next(iter(dataloader))
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for i in range(min(num_samples, len(images))):
        img = images[i].permute(1, 2, 0).numpy()
        target = targets[i]
        
        # Original image
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Image {i+1}")
        axes[i, 0].axis('off')
        
        # Image with bounding boxes
        axes[i, 1].imshow(img)
        for box in target['boxes']:
            x1, y1, x2, y2 = box
            rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                     linewidth=2, edgecolor='r', facecolor='none')
            axes[i, 1].add_patch(rect)
        axes[i, 1].set_title(f"Image {i+1} with Boxes (n={len(target['boxes'])})")
        axes[i, 1].axis('off')
        
        # Show all instance masks combined
        if len(target['masks']) > 0:
            combined_mask = target['masks'].sum(dim=0).numpy()
        else:
            combined_mask = np.zeros((img.shape[0], img.shape[1]))
        axes[i, 2].imshow(combined_mask, cmap='gray')
        axes[i, 2].set_title(f"Instances: {len(target['masks'])}")
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# If the TreeDataset class code is already in your notebook, just call:
preprocessed_folder = "/kaggle/working/preprocessed/"
train_loader, val_loader = get_dataloaders(preprocessed_folder, batch_size=1, train_split=0.9)

# Visualize samples
visualize_samples(train_loader, num_samples=1)

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import numpy as np
from tqdm import tqdm
import json
from collections import OrderedDict
from torch.cuda.amp import autocast, GradScaler
import random
from PIL import Image

In [ ]:
import os
import torch
import torch.nn as nn
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import numpy as np
from tqdm import tqdm
import json
from torch.utils.data import DataLoader

# Import your dataset class (assuming it's in the same directory)
# from your_dataset_file import TreeDataset, get_dataloaders

class SwinBackbone(nn.Module):
    """
    Swin Transformer V2 backbone wrapper for Mask R-CNN
    Uses timm library for easy access to pretrained Swin models
    """
    def __init__(self, model_name='swinv2_tiny_window16_256', pretrained=True):
        super().__init__()
        try:
            import timm
        except ImportError:
            raise ImportError("Please install timm: pip install timm")
        
        # Load Swin Transformer V2 from timm with img_size parameter
        self.swin = timm.create_model(
            model_name,
            pretrained=pretrained,
            features_only=True,  
            out_indices=(0, 1, 2, 3),  
            img_size=1024  
        )
        
        # Get feature dimensions for FPN
        self.out_channels = self.swin.feature_info.channels()
        print(f"Swin feature channels: {self.out_channels}")
        
    def forward(self, x):
        features = self.swin(x)
        # Return as OrderedDict for FPN compatibility
        out = {}
        for idx, feat in enumerate(features):
            out[f'feat{idx}'] = feat
        return out


def get_swin_maskrcnn_model(num_classes=2, model_name='swinv2_tiny_window16_256', 
                            pretrained_backbone=True, min_size=1024, max_size=1024):
    """
    Create Mask R-CNN model with Swin Transformer V2 backbone
    
    Args:
        num_classes: Number of classes (background + object classes)
        model_name: Swin model variant from timm
        pretrained_backbone: Use pretrained Swin weights
        min_size: Minimum size of the image to be rescaled before feeding to backbone
        max_size: Maximum size of the image to be rescaled before feeding to backbone
    
    Returns:
        model: Mask R-CNN model with Swin backbone
    """
    
    # Start with standard Mask R-CNN (we'll replace the backbone)
    model = maskrcnn_resnet50_fpn(
        pretrained=False, 
        pretrained_backbone=False,
        min_size=min_size,
        max_size=max_size
    )
    
    # Replace backbone with Swin Transformer
    swin_backbone = SwinBackbone(model_name, pretrained_backbone)
    
    # Create FPN with Swin features
    from torchvision.ops.feature_pyramid_network import FeaturePyramidNetwork, LastLevelMaxPool
    from torchvision.ops import misc as misc_nn_ops
    
    # Get channel dimensions from Swin
    in_channels_list = swin_backbone.out_channels
    out_channels = 256
    
    print(f"Building FPN with input channels: {in_channels_list}")
    
    # Build FPN with extra pooling level to match RPN expectations (5 levels)
    fpn = FeaturePyramidNetwork(
        in_channels_list=in_channels_list,
        out_channels=out_channels,
        extra_blocks=LastLevelMaxPool()  # Adds 5th level via max pooling
    )
    
    # Combine Swin + FPN as backbone
    class SwinFPNBackbone(nn.Module):
        def __init__(self, swin, fpn):
            super().__init__()
            self.body = swin
            self.fpn = fpn
            self.out_channels = out_channels
            
        def forward(self, x):
            features = self.body(x)
            # Convert dict to OrderedDict with proper keys
            from collections import OrderedDict
            
            # Swin outputs features in (B, H, W, C) format
            # Need to permute to (B, C, H, W) for FPN
            features_dict = OrderedDict()
            for i, feat in enumerate(features.values()):
                # Check if permutation is needed
                if feat.dim() == 4 and feat.shape[-1] in self.body.out_channels:
                    # Permute from (B, H, W, C) to (B, C, H, W)
                    feat = feat.permute(0, 3, 1, 2).contiguous()
                features_dict[str(i)] = feat
            
            fpn_features = self.fpn(features_dict)
            return fpn_features
    
    # Replace the model's backbone
    model.backbone = SwinFPNBackbone(swin_backbone, fpn)
    
    # Replace the box predictor
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    # Replace the mask predictor
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )
    
    print(f"✓ Model created successfully with {model_name}")
    return model




In [ ]:
# def train_one_epoch(model, optimizer, data_loader, device, epoch):
#     """Train for one epoch"""
#     model.train()
#     epoch_loss = 0
#     num_batches = 0
    
#     progress_bar = tqdm(data_loader, desc=f'Epoch {epoch}')
    
#     for images, targets in progress_bar:
#         # Skip empty batches
#         if len(images) == 0:
#             continue
            
#         # Move to device
#         images = [img.to(device) for img in images]
#         targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
#         # Skip if any target has no boxes
#         valid = all(len(t['boxes']) > 0 for t in targets)
#         if not valid:
#             continue
        
#         try:
#             # Forward pass
#             loss_dict = model(images, targets)
#             losses = sum(loss for loss in loss_dict.values())
            
#             # Check for NaN
#             if torch.isnan(losses):
#                 print("Warning: NaN loss detected, skipping batch")
#                 continue
            
#             # Backward pass
#             optimizer.zero_grad()
#             losses.backward()
            
#             # Gradient clipping to prevent exploding gradients
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
#             optimizer.step()
            
#             # Update progress
#             epoch_loss += losses.item()
#             num_batches += 1
#             progress_bar.set_postfix({'loss': losses.item()})
            
#         except Exception as e:
#             print(f"Error in batch: {e}")
#             continue
    
#     return epoch_loss / max(num_batches, 1)


# @torch.no_grad()
# def evaluate(model, data_loader, device):
#     """Evaluate model"""
#     model.eval()
#     total_loss = 0
#     num_batches = 0
    
#     for images, targets in tqdm(data_loader, desc='Evaluating'):
#         if len(images) == 0:
#             continue
            
#         images = [img.to(device) for img in images]
#         targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
#         # Skip if any target has no boxes
#         valid = all(len(t['boxes']) > 0 for t in targets)
#         if not valid:
#             continue
        
#         try:
#             # Get predictions - need train mode for losses
#             model.train()
#             loss_dict = model(images, targets)
#             losses = sum(loss for loss in loss_dict.values())
            
#             if not torch.isnan(losses):
#                 total_loss += losses.item()
#                 num_batches += 1
#         except Exception as e:
#             print(f"Error in evaluation batch: {e}")
#             continue
    
#     model.eval()
#     return total_loss / max(num_batches, 1)


# def train_model(model, train_loader, val_loader, num_epochs=25, 
#                 lr=0.0001, device='cuda', save_dir='./checkpoints'):
#     """
#     Main training loop
    
#     Args:
#         model: Mask R-CNN model
#         train_loader: Training data loader
#         val_loader: Validation data loader
#         num_epochs: Number of training epochs
#         lr: Learning rate
#         device: Device to train on
#         save_dir: Directory to save checkpoints
#     """
#     os.makedirs(save_dir, exist_ok=True)
    
#     # Move model to device
#     model.to(device)
    
#     # Optimizer
#     params = [p for p in model.parameters() if p.requires_grad]
#     #optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=0.0001)
#     optimizer = torch.optim.AdamW(
#     params, 
#     lr=lr, 
#     weight_decay=0.01,  
#     betas=(0.9, 0.999),
#     eps=1e-8
# )
    
#     # Learning rate scheduler
#     lr_scheduler = torch.optim.lr_scheduler.StepLR(
#         optimizer, step_size=5, gamma=0.5
#     )
    
#     # Training history
#     history = {
#         'train_loss': [],
#         'val_loss': [],
#     }
    
#     best_val_loss = float('inf')
    
#     for epoch in range(1, num_epochs + 1):
#         print(f'\n{"="*50}')
#         print(f'Epoch {epoch}/{num_epochs}')
#         print(f'{"="*50}')
        
#         # Train
#         train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
#         history['train_loss'].append(train_loss)
#         print(f'Train Loss: {train_loss:.4f}')
        
#         # Validate
#         val_loss = evaluate(model, val_loader, device)
#         history['val_loss'].append(val_loss)
#         print(f'Val Loss: {val_loss:.4f}')
        
#         # Update learning rate
#         current_lr = optimizer.param_groups[0]['lr']
#         print(f'Learning Rate: {current_lr:.6f}')
#         lr_scheduler.step()
        
#         # Save best model
#         if val_loss < best_val_loss:
#             best_val_loss = val_loss
#             torch.save({
#                 'epoch': epoch,
#                 'model_state_dict': model.state_dict(),
#                 'optimizer_state_dict': optimizer.state_dict(),
#                 'val_loss': val_loss,
#                 'train_loss': train_loss,
#             }, os.path.join(save_dir, 'best_model.pth'))
#             print(f'✓ Saved best model (val_loss: {val_loss:.4f})')
        

#     # Save final model
#     torch.save({
#         'epoch': num_epochs,
#         'model_state_dict': model.state_dict(),
#         'optimizer_state_dict': optimizer.state_dict(),
#     }, os.path.join(save_dir, 'final_model.pth'))
    
#     # Save training history
#     with open(os.path.join(save_dir, 'history.json'), 'w') as f:
#         json.dump(history, f, indent=2)
    
#     print(f'\n{"="*50}')
#     print('Training Complete!')
#     print(f'Best validation loss: {best_val_loss:.4f}')
#     print(f'{"="*50}')
    
#     return model, history


# # ===== MAIN EXECUTION =====
# if __name__ == '__main__':
#     # Configuration
#     PREPROCESSED_FOLDER = "/kaggle/working/preprocessed/"
#     BATCH_SIZE = 4
#     NUM_EPOCHS = 50
#     LEARNING_RATE = 0.00001
#     NUM_CLASSES = 2  # background + tree
#     DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
#     # Swin model variants (choose one):
#     # 'swinv2_tiny_window16_256' - Smallest, fastest (~28M params)
#     # 'swinv2_small_window16_256' - Balanced (~50M params)
#     # 'swinv2_base_window16_256' - Larger, better accuracy (~88M params)
#     MODEL_NAME = 'swinv2_tiny_window16_256'
    
#     print(f'{"="*50}')
#     print('Swin Transformer V2 Instance Segmentation Training')
#     print(f'{"="*50}')
#     print(f'Device: {DEVICE}')
#     print(f'Model: {MODEL_NAME}')
#     print(f'Batch Size: {BATCH_SIZE}')
#     print(f'Epochs: {NUM_EPOCHS}')
#     print(f'Learning Rate: {LEARNING_RATE}')
#     print(f'{"="*50}\n')
    
  
#     print(f'✓ Train samples: {len(train_loader.dataset)}')
#     print(f'✓ Val samples: {len(val_loader.dataset)}')
#     print(f'✓ Train batches: {len(train_loader)}')
#     print(f'✓ Val batches: {len(val_loader)}\n')
    
#     # Create model
#     print('Creating Swin Transformer V2 Mask R-CNN model...')
#     model = get_swin_maskrcnn_model(
#         num_classes=NUM_CLASSES,
#         model_name=MODEL_NAME,
#         pretrained_backbone=True,
#         min_size=1024, 
#         max_size=1024
#     )
#     checkpoint = torch.load("/kaggle/input/best-swin-v2/pytorch/default/1/best_model.pth", map_location="cuda")
#     model.load_state_dict(checkpoint['model_state_dict'])
#     model.to("cuda")
#     print()
    


In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    """Train for one epoch"""
    model.train()
    epoch_loss = 0
    num_batches = 0
    
    progress_bar = tqdm(data_loader, desc=f'Epoch {epoch}')
    
    for images, targets in progress_bar:
        # Skip empty batches
        if len(images) == 0:
            continue
            
        # Move to device
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Skip if any target has no boxes
        valid = all(len(t['boxes']) > 0 for t in targets)
        if not valid:
            continue
        
        try:
            # Forward pass
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            
            # Check for NaN
            if torch.isnan(losses):
                print("Warning: NaN loss detected, skipping batch")
                continue
            
            # Backward pass
            optimizer.zero_grad()
            losses.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            # Update progress
            epoch_loss += losses.item()
            num_batches += 1
            progress_bar.set_postfix({'loss': losses.item()})
            
        except Exception as e:
            print(f"Error in batch: {e}")
            continue
    
    return epoch_loss / max(num_batches, 1)


@torch.no_grad()
def evaluate(model, data_loader, device):
    """Evaluate model"""
    model.eval()
    total_loss = 0
    num_batches = 0
    
    for images, targets in tqdm(data_loader, desc='Evaluating'):
        if len(images) == 0:
            continue
            
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Skip if any target has no boxes
        valid = all(len(t['boxes']) > 0 for t in targets)
        if not valid:
            continue
        
        try:
            # Get predictions - need train mode for losses
            model.train()
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            
            if not torch.isnan(losses):
                total_loss += losses.item()
                num_batches += 1
        except Exception as e:
            print(f"Error in evaluation batch: {e}")
            continue
    
    model.eval()
    return total_loss / max(num_batches, 1)


def train_model(model, train_loader, val_loader, num_epochs=25, 
                lr=0.0001, device='cuda', save_dir='./checkpoints',
                patience=5, min_delta=0.001):
    """
    Main training loop with early stopping and ReduceLROnPlateau
    
    Args:
        model: Mask R-CNN model
        train_loader: Training data loader
        val_loader: Validation data loader
        num_epochs: Maximum number of training epochs
        lr: Initial learning rate
        device: Device to train on
        save_dir: Directory to save checkpoints
        patience: Number of epochs without improvement before stopping
        min_delta: Minimum change to qualify as improvement
    
    Returns:
        model, history
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Move model to device
    model.to(device)
    
    # Optimizer with increased weight decay for more regularization
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(
        params, 
        lr=lr, 
        weight_decay=0.1,  # Increased from 0.01 for stronger regularization
        betas=(0.9, 0.999),
        eps=1e-8
    )
    
    # Learning rate scheduler - Reduce on plateau
    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min', 
        factor=0.5, 
        patience=3, 
        min_lr=1e-7,
        verbose=True
    )
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
    }
    
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    
    for epoch in range(1, num_epochs + 1):
        print(f'\n{"="*50}')
        print(f'Epoch {epoch}/{num_epochs}')
        print(f'{"="*50}')
        
        # Train
        train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
        history['train_loss'].append(train_loss)
        print(f'Train Loss: {train_loss:.4f}')
        
        # Validate
        val_loss = evaluate(model, val_loader, device)
        history['val_loss'].append(val_loss)
        print(f'Val Loss: {val_loss:.4f}')
        
        # Update learning rate scheduler
        lr_scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Learning Rate: {current_lr:.6f}')
        
        # Check for improvement
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'train_loss': train_loss,
            }, os.path.join(save_dir, 'best_model.pth'))
            print(f'✓ Saved best model (val_loss: {val_loss:.4f})')
        else:
            epochs_without_improvement += 1
            print(f'No improvement ({epochs_without_improvement}/{patience})')
        
        # Early stopping
        if epochs_without_improvement >= patience:
            print(f'\nEarly stopping triggered after {epoch} epochs')
            print(f'Best validation loss: {best_val_loss:.4f}')
            break

    # Save final model
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, os.path.join(save_dir, 'final_model.pth'))
    
    # Save training history
    with open(os.path.join(save_dir, 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)
    
    print(f'\n{"="*50}')
    print('Training Complete!')
    print(f'Best validation loss: {best_val_loss:.4f}')
    print(f'{"="*50}')
    
    return model, history


In [ ]:
# ===== MAIN EXECUTION =====
if __name__ == '__main__':
    # Configuration
    PREPROCESSED_FOLDER = "/kaggle/working/preprocessed/"
    BATCH_SIZE = 6
    NUM_EPOCHS = 80
    LEARNING_RATE = 0.00001
    NUM_CLASSES = 2  # background + tree
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Swin model variants (choose one):
    # 'swinv2_tiny_window16_256' - Smallest, fastest (~28M params)
    # 'swinv2_small_window16_256' - Balanced (~50M params)
    # 'swinv2_base_window16_256' - Larger, better accuracy (~88M params)
    MODEL_NAME = 'swinv2_tiny_window16_256'
    
    print(f'{"="*50}')
    print('Swin Transformer V2 Instance Segmentation Training')
    print(f'{"="*50}')
    print(f'Device: {DEVICE}')
    print(f'Model: {MODEL_NAME}')
    print(f'Batch Size: {BATCH_SIZE}')
    print(f'Epochs: {NUM_EPOCHS}')
    print(f'Learning Rate: {LEARNING_RATE}')
    print(f'{"="*50}\n')
    
  
    print(f'✓ Train samples: {len(train_loader.dataset)}')
    print(f'✓ Val samples: {len(val_loader.dataset)}')
    print(f'✓ Train batches: {len(train_loader)}')
    print(f'✓ Val batches: {len(val_loader)}\n')
    
    # Create model
    print('Creating Swin Transformer V2 Mask R-CNN model...')
    model = get_swin_maskrcnn_model(
        num_classes=NUM_CLASSES,
        model_name=MODEL_NAME,
        pretrained_backbone=True,
        min_size=1024, 
        max_size=1024
    )
    checkpoint = torch.load("/kaggle/input/best-swin-v2/pytorch/default/1/best_model.pth", map_location="cuda")
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to("cuda")
    print()

In [ ]:
    # Train model
    print('Starting training...')
    trained_model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=50,
    lr=1e-4,
    device="cuda",
    save_dir='./swin_checkpoints',
    patience=5,
    min_delta=0.001
)


    
    # Plot training curves
    try:
        import matplotlib.pyplot as plt
        
        plt.figure(figsize=(10, 5))
        plt.plot(history['train_loss'], label='Train Loss')
        plt.plot(history['val_loss'], label='Val Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training History')
        plt.legend()
        plt.grid(True)
        plt.savefig('./swin_checkpoints/training_curves.png', dpi=150, bbox_inches='tight')
        print('\n✓ Training curves saved to ./swin_checkpoints/training_curves.png')
    except Exception as e:
        print(f'Could not plot training curves: {e}')

In [ ]:
import os
import torch
import numpy as np
import rasterio
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import json

# ---- CONFIG ----
CHECKPOINT_PATH = "/kaggle/working/swin_checkpoints/best_model.pth"
MODEL_NAME = 'swinv2_tiny_window16_256'
NUM_CLASSES = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PATCH_SIZE = 1024
STRIDE = PATCH_SIZE // 2
RESIZE_FACTOR = 0.6
SCORE_THRESHOLD = 0.9

def load_full_model(checkpoint_path, model_name, num_classes, device):
    model = get_swin_maskrcnn_model(
        num_classes=num_classes,
        model_name=model_name,
        pretrained_backbone=False,
        min_size=PATCH_SIZE,
        max_size=PATCH_SIZE
    )
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device).eval()
    return model

# ---- Helper: Simple NMS to merge overlapping predictions across patches ----
def merge_predictions(all_preds, iou_threshold=0.5):
    """
    Merges predictions from overlapping patches using Non-Maximum Suppression.
    Returns a dictionary with merged boxes, scores, and masks.
    """
    if len(all_preds['boxes']) == 0:
        return {'boxes': [], 'scores': [], 'masks': []}
    
    # Convert to numpy arrays
    boxes = np.array(all_preds['boxes'])
    scores = np.array(all_preds['scores'])
    masks = all_preds['masks'] # List of (H, W) arrays
    
    # Sort by score descending
    sorted_indices = np.argsort(scores)[::-1]
    keep = []
    suppressed = set()
    
    for i in sorted_indices:
        if i in suppressed:
            continue
        keep.append(i)
        # Suppress other boxes with high IoU
        for j in sorted_indices:
            if j == i or j in suppressed:
                continue
            iou = calculate_iou(boxes[i], boxes[j])
            if iou >= iou_threshold:
                suppressed.add(j)
    
    merged_boxes = [boxes[i] for i in keep]
    merged_scores = [scores[i] for i in keep]
    merged_masks = [masks[i] for i in keep]
    
    return {
        'boxes': merged_boxes,
        'scores': merged_scores,
        'masks': merged_masks
    }

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    if x2 <= x1 or y2 <= y1:
        return 0.0
    
    inter = (x2 - x1) * (y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0.0

# ---- Helper: Create GT mask and boxes from LabelMe JSON ----
def load_ground_truth(tiff_path, annotation_folder=None):
    """
    Load ground truth polygons, boxes, and create an instance mask from the .json file.
    Assumes the .json filename matches the .tif filename.
    """
    if annotation_folder is None:
        annotation_folder = os.path.dirname(tiff_path)
    base_name = os.path.splitext(os.path.basename(tiff_path))[0]
    json_path = os.path.join(annotation_folder, f"{base_name}.json")
    print(f"Loading ground truth from: {json_path}")
    
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Extract polygons
    polygons = []
    boxes = []
    for shape in data.get("shapes", []):
        if shape["shape_type"] != "polygon":
            continue
        pts = shape["points"]
        if len(pts) < 3:
            continue
        polygons.append(pts)
        xs = [p[0] for p in pts]
        ys = [p[1] for p in pts]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        boxes.append([x_min, y_min, x_max, y_max])
    
    print(f"Loaded {len(polygons)} ground truth instances.")
    return polygons, boxes

def create_gt_instance_mask(image_shape, polygons):
    """
    Create a single-channel mask where each instance has a unique ID (1, 2, 3...).
    """
    h, w = image_shape
    mask = np.zeros((h, w), dtype=np.int32)
    
    for idx, poly in enumerate(polygons, start=1):
        pts = np.array(poly, dtype=np.int32)
        # Clip points to image bounds
        pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)
        cv2.fillPoly(mask, [pts], idx)
    
    return mask

# ---- Full Image Inference ----
def infer_on_full_tiff(tiff_path, model, device):
    print(f"Loading full image: {tiff_path}")
    with rasterio.open(tiff_path) as src:
        image = np.moveaxis(src.read(), 0, -1) # CHW -> HWC
    
    H_orig, W_orig = image.shape[:2]
    new_H, new_W = int(H_orig * RESIZE_FACTOR), int(W_orig * RESIZE_FACTOR)
    image_resized = cv2.resize(image, (new_W, new_H), interpolation=cv2.INTER_LINEAR)
    
    # Pad to multiples of PATCH_SIZE
    pad_h = (int(np.ceil(new_H / PATCH_SIZE)) * PATCH_SIZE) - new_H
    pad_w = (int(np.ceil(new_W / PATCH_SIZE)) * PATCH_SIZE) - new_W
    image_padded = np.pad(image_resized, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
    H_pad, W_pad = image_padded.shape[:2]
    
    # Create empty canvases for merged results
    merged_boxes = []
    merged_scores = []
    merged_masks = []
    
    print(f"Padded size: {H_pad}x{W_pad}. Running inference on patches...")
    
    for y in range(0, H_pad - PATCH_SIZE + 1, STRIDE):
        for x in range(0, W_pad - PATCH_SIZE + 1, STRIDE):
            patch = image_padded[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            # Convert to tensor
            patch_tensor = torch.from_numpy(patch).permute(2, 0, 1).float().to(device) / 255.0
            patch_tensor = patch_tensor.unsqueeze(0) # batch dim
            
            with torch.no_grad():
                pred = model(patch_tensor)[0]
            
            keep = pred['scores'] > SCORE_THRESHOLD
            if not keep.any():
                continue
            
            # Transform predictions to full-image coordinates
            boxes = pred['boxes'][keep].cpu().numpy()
            scores = pred['scores'][keep].cpu().numpy()
            masks = pred['masks'][keep].cpu().numpy() # [N, 1, H, W]
            
            for i in range(len(boxes)):
                # Adjust box coordinates
                box = boxes[i].copy()
                box[[0, 2]] += x # x1, x2
                box[[1, 3]] += y # y1, y2
                merged_boxes.append(box)
                merged_scores.append(scores[i])
                
                # Create a full-size mask for this instance
                full_mask_instance = np.zeros((H_pad, W_pad), dtype=np.float32)
                mask_bin = (masks[i, 0] > 0.5).astype(np.float32)
                full_mask_instance[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = mask_bin
                merged_masks.append(full_mask_instance)
    
    # Merge overlapping predictions
    print("Merging predictions across patches...")
    final_preds = merge_predictions({
        'boxes': merged_boxes,
        'scores': merged_scores,
        'masks': merged_masks
    }, iou_threshold=0.5)
    
    # Crop back to resized (non-padded) region
    final_boxes = [box for box in final_preds['boxes']]
    final_scores = [score for score in final_preds['scores']]
    final_masks = [mask[:new_H, :new_W] for mask in final_preds['masks']]
    
    # Convert boxes back to original image scale
    scale_back = 1.0 / RESIZE_FACTOR
    for i, box in enumerate(final_boxes):
        final_boxes[i] = [coord * scale_back for coord in box]
    
    return final_boxes, final_scores, final_masks, image

# ---- Visualization: 6-Panel Grid with GT ----
def visualize_full_image_comparison(original_image, predicted_boxes, predicted_scores, predicted_masks, gt_boxes, gt_polygons, save_path=None):
    """
    Creates a 6-panel visualization comparing predictions and ground truth on the full original image.
    """
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    # --- TOP ROW: GROUND TRUTH ---
    # Panel 1: Original Image
    axes[0, 0].imshow(original_image)
    axes[0, 0].set_title("Original Image", fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Panel 2: Ground Truth Boxes
    axes[0, 1].imshow(original_image)
    for box in gt_boxes:
        x1, y1, x2, y2 = map(int, box)
        rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
        axes[0, 1].add_patch(rect)
    axes[0, 1].set_title(f"Ground Truth Boxes (n={len(gt_boxes)})", fontsize=14, fontweight='bold', color='green')
    axes[0, 1].axis('off')
    
    # Panel 3: Ground Truth Masks
    H_orig, W_orig = original_image.shape[:2]
    if len(gt_polygons) > 0:
        # Create GT instance mask
        gt_mask = create_gt_instance_mask((H_orig, W_orig), gt_polygons)
        # Use colormap for visualization
        cmap = plt.get_cmap('tab20')
        colored_gt_mask = cmap(gt_mask / gt_mask.max())[:, :, :3] if gt_mask.max() > 0 else np.zeros_like(original_image, dtype=np.float32)
        colored_gt_mask[gt_mask == 0] = [0, 0, 0]
        axes[0, 2].imshow(colored_gt_mask)
    else:
        axes[0, 2].imshow(np.zeros_like(original_image[:, :, 0]), cmap='gray')
    axes[0, 2].set_title(f"Ground Truth Masks (n={len(gt_polygons)})", fontsize=14, fontweight='bold', color='green')
    axes[0, 2].axis('off')
    
    # --- BOTTOM ROW: PREDICTIONS ---
    # Panel 4: Original Image
    axes[1, 0].imshow(original_image)
    axes[1, 0].set_title("Original Image", fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')
    
    # Panel 5: Predicted Boxes
    axes[1, 1].imshow(original_image)
    for box in predicted_boxes:
        x1, y1, x2, y2 = map(int, box)
        rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
        axes[1, 1].add_patch(rect)
    axes[1, 1].set_title(f"Predicted Boxes (n={len(predicted_boxes)})", fontsize=14, fontweight='bold', color='red')
    axes[1, 1].axis('off')
    
    # Panel 6: Predicted Masks
    H_orig, W_orig = original_image.shape[:2]
    if len(predicted_masks) > 0:
        # Create an empty mask canvas matching the original image size
        combined_mask = np.zeros((H_orig, W_orig), dtype=np.uint8)
        # For each instance mask, resize it from the resized/padded coordinate space back to original image space
        for i, mask_resized_padded in enumerate(predicted_masks):
            # Resize the mask back to original image dimensions
            mask_original_size = cv2.resize(
                mask_resized_padded.astype(np.float32),
                (W_orig, H_orig),
                interpolation=cv2.INTER_NEAREST # Use nearest neighbor to preserve instance IDs
            )
            # Assign a unique value for this instance (e.g., i+1)
            combined_mask[mask_original_size > 0] = i + 1
        
        # Normalize to 0-255 for display
        if combined_mask.max() > 0:
            # Use a colormap to assign colors to each instance ID
            cmap = plt.get_cmap('tab20')
            colored_mask = cmap(combined_mask / combined_mask.max())[:, :, :3]
            colored_mask[combined_mask == 0] = [0, 0, 0]
            axes[1, 2].imshow(colored_mask)
        else:
            axes[1, 2].imshow(np.zeros_like(original_image[:, :, 0]), cmap='gray')
    else:
        axes[1, 2].imshow(np.zeros_like(original_image[:, :, 0]), cmap='gray')
    axes[1, 2].set_title(f"Predicted Masks (n={len(predicted_masks)})", fontsize=14, fontweight='bold', color='red')
    axes[1, 2].axis('off')
    
    # Add overall title
    fig.suptitle(f"Full-Image Instance Segmentation: Prediction vs Ground Truth", fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Saved visualization to: {save_path}")
    
    plt.show()

# ---- MAIN ----
if __name__ == "__main__":
    # Load model
    model = load_full_model(CHECKPOINT_PATH, MODEL_NAME, NUM_CLASSES, DEVICE)
    
    # Example: evaluate on one full image
    print("Model hasnt seen- /kaggle/input/tree-data-for-instance-segmentation/tree_data_for_instance_segmentation/McCormick_25Apr01_4_4_SKIP.tif image")
    tiff_file = "/kaggle/input/tree-data-for-instance-segmentation/tree_data_for_instance_segmentation/McCormick_25Apr01_4_2.tif"
    
    # Load Ground Truth
    gt_polygons, gt_boxes = load_ground_truth(tiff_file)
    
    # Run inference
    boxes, scores, masks, orig_img = infer_on_full_tiff(tiff_file, model, DEVICE)
    
    # Visualize
    visualize_full_image_comparison(orig_img, boxes, scores, masks, gt_boxes, gt_polygons, save_path="./full_image_prediction_with_gt_6panel.png")
    
    print(f"✅ Done! Found {len(boxes)} fallen trees in the full image.")

In [ ]:
import os
import torch
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random

# ---- CONFIG ----
CHECKPOINT_PATH = "/kaggle/working/swin_checkpoints/best_model.pth"
MODEL_NAME = 'swinv2_tiny_window16_256'
NUM_CLASSES = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PATCH_SIZE = 1024
SCORE_THRESHOLD = 0.85

def load_model(checkpoint_path, model_name, num_classes, device):
    """Load the trained model"""
    model = get_swin_maskrcnn_model(
        num_classes=num_classes,
        model_name=model_name,
        pretrained_backbone=False,
        min_size=PATCH_SIZE,
        max_size=PATCH_SIZE
    )
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device).eval()
    return model

def sharpen_mask(mask, sigma=3.0, amount=1.5):
    """Sharpen the mask using unsharp masking."""
    # Ensure mask is float32
    mask = mask.astype(np.float32)
    blurred = cv2.GaussianBlur(mask, (0, 0), sigmaX=sigma)
    sharpened = mask + amount * (mask - blurred)
    sharpened = np.clip(sharpened, 0, 1)
    return sharpened

def visualize_prediction_vs_gt(image, prediction, ground_truth, idx, save_path=None):
    """
    Creates a 6-panel visualization comparing predictions and ground truth.
    
    Args:
        image: tensor of shape (3, H, W) normalized to [0, 1]
        prediction: dict with 'boxes', 'scores', 'masks'
        ground_truth: dict with 'boxes', 'masks'
        idx: sample index for title
        save_path: optional path to save the figure
    """
    # Convert image tensor to numpy
    img_np = image.cpu().permute(1, 2, 0).numpy()  # (H, W, 3)
    img_np = np.clip(img_np, 0, 1)  # Ensure values are in [0, 1]
    
    # Extract predictions
    pred_boxes = prediction['boxes'].cpu().numpy()
    pred_scores = prediction['scores'].cpu().numpy()
    pred_masks = prediction['masks'].cpu().numpy()  # (N, 1, H, W)
    
    # Extract ground truth
    gt_boxes = ground_truth['boxes'].cpu().numpy()
    gt_masks = ground_truth['masks'].cpu().numpy()  # (N, H, W)
    
    # Create figure with 2 rows, 3 columns
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    # --- TOP ROW: GROUND TRUTH ---
    # Panel 1: Original Image
    axes[0, 0].imshow(img_np)
    axes[0, 0].set_title("Original Image", fontsize=14, fontweight='bold')
    axes[0, 0].axis('off')
    
    # Panel 2: Ground Truth Boxes
    axes[0, 1].imshow(img_np)
    for box in gt_boxes:
        x1, y1, x2, y2 = box
        rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                 linewidth=2, edgecolor='lime', facecolor='none')
        axes[0, 1].add_patch(rect)
    axes[0, 1].set_title(f"Ground Truth Boxes (n={len(gt_boxes)})", 
                        fontsize=14, fontweight='bold', color='green')
    axes[0, 1].axis('off')
    
    # Panel 3: Ground Truth Masks
    if len(gt_masks) > 0:
        # Combine all GT masks with different colors
        combined_gt_mask = np.zeros(gt_masks.shape[1:], dtype=np.int32)
        for i, mask in enumerate(gt_masks, start=1):
            combined_gt_mask[mask > 0] = i
        
        # Use colormap for visualization
        if combined_gt_mask.max() > 0:
            cmap = plt.get_cmap('tab20')
            colored_gt_mask = cmap(combined_gt_mask / combined_gt_mask.max())[:, :, :3]
            colored_gt_mask[combined_gt_mask == 0] = [0, 0, 0]
            axes[0, 2].imshow(colored_gt_mask)
        else:
            axes[0, 2].imshow(np.zeros_like(img_np[:, :, 0]), cmap='gray')
    else:
        axes[0, 2].imshow(np.zeros_like(img_np[:, :, 0]), cmap='gray')
    axes[0, 2].set_title(f"Ground Truth Masks (n={len(gt_masks)})", 
                        fontsize=14, fontweight='bold', color='green')
    axes[0, 2].axis('off')
    
    # --- BOTTOM ROW: PREDICTIONS ---
    # Panel 4: Original Image
    axes[1, 0].imshow(img_np)
    axes[1, 0].set_title("Original Image", fontsize=14, fontweight='bold')
    axes[1, 0].axis('off')
    
    # Panel 5: Predicted Boxes
    axes[1, 1].imshow(img_np)
    for box, score in zip(pred_boxes, pred_scores):
        x1, y1, x2, y2 = box
        rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1, 
                                 linewidth=2, edgecolor='red', facecolor='none')
        axes[1, 1].add_patch(rect)
        # Add score text
        axes[1, 1].text(x1, y1-5, f'{score:.2f}', 
                       color='red', fontsize=8, fontweight='bold',
                       bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))
    axes[1, 1].set_title(f"Predicted Boxes (n={len(pred_boxes)})", 
                        fontsize=14, fontweight='bold', color='red')
    axes[1, 1].axis('off')
    
    # Panel 6: Predicted Masks
    if len(pred_masks) > 0:
        # Combine all predicted masks with different colors
        combined_pred_mask = np.zeros(pred_masks.shape[2:], dtype=np.int32)
        for i, mask in enumerate(pred_masks, start=1):
            mask_prob = mask[0]  # (H, W)
            sharpened_prob = sharpen_mask(mask_prob)
            mask_bin = (sharpened_prob > 0.5).astype(np.uint8)
            combined_pred_mask[mask_bin > 0] = i
        
        # Use colormap for visualization
        if combined_pred_mask.max() > 0:
            cmap = plt.get_cmap('tab20')
            colored_pred_mask = cmap(combined_pred_mask / combined_pred_mask.max())[:, :, :3]
            colored_pred_mask[combined_pred_mask == 0] = [0, 0, 0]
            axes[1, 2].imshow(colored_pred_mask)
        else:
            axes[1, 2].imshow(np.zeros_like(img_np[:, :, 0]), cmap='gray')
    else:
        axes[1, 2].imshow(np.zeros_like(img_np[:, :, 0]), cmap='gray')
    axes[1, 2].set_title(f"Predicted Masks (n={len(pred_masks)})", 
                        fontsize=14, fontweight='bold', color='red')
    axes[1, 2].axis('off')
    
    # Add overall title
    fig.suptitle(f"Sample {idx}: Prediction vs Ground Truth", 
                fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Saved visualization to: {save_path}")
    
    plt.show()


def visualize_random_validation_samples(val_loader, model, device, num_samples=5, save_dir=None):
    """
    Randomly select and visualize samples from the validation loader.
    
    Args:
        val_loader: validation DataLoader
        model: trained model
        device: torch device
        num_samples: number of random samples to visualize
        save_dir: optional directory to save visualizations
    """
    model.eval()
    
    # Get all samples from validation loader
    all_images = []
    all_targets = []
    
    print("Collecting validation samples...")
    for images, targets in val_loader:
        all_images.extend(images)
        all_targets.extend(targets)
    
    print(f"Total validation samples: {len(all_images)}")
    
    # Randomly select samples
    num_samples = min(num_samples, len(all_images))
    random_indices = random.sample(range(len(all_images)), num_samples)
    
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
    
    print(f"\nVisualizing {num_samples} random samples...")
    
    for i, idx in enumerate(random_indices, 1):
        print(f"\nProcessing sample {i}/{num_samples} (index {idx})...")
        
        image = all_images[idx].to(device)
        target = all_targets[idx]
        
        # Run inference
        with torch.no_grad():
            image_batch = image.unsqueeze(0)  # Add batch dimension
            predictions = model(image_batch)[0]
        
        # Filter predictions by score threshold
        keep = predictions['scores'] > SCORE_THRESHOLD
        filtered_pred = {
            'boxes': predictions['boxes'][keep],
            'scores': predictions['scores'][keep],
            'masks': predictions['masks'][keep]
        }
        
        print(f"  GT instances: {len(target['boxes'])}")
        print(f"  Predicted instances (score > {SCORE_THRESHOLD}): {len(filtered_pred['boxes'])}")
        
        # Create save path if needed
        save_path = None
        if save_dir:
            save_path = os.path.join(save_dir, f"val_sample_{idx}.png")
        
        # Visualize
        visualize_prediction_vs_gt(image, filtered_pred, target, idx, save_path)


# ---- MAIN ----
if __name__ == "__main__":
  
    
    # Load model
    print(f"Loading model from {CHECKPOINT_PATH}...")
    model = load_model(CHECKPOINT_PATH, MODEL_NAME, NUM_CLASSES, DEVICE)
    print("✓ Model loaded successfully")
    
    # Visualize 5 random validation samples
    visualize_random_validation_samples(
        val_loader=val_loader,
        model=model,
        device=DEVICE,
        num_samples=5,
        save_dir="/kaggle/working/val_visualizations"
    )
    
    print("\n✅ Done! Visualizations complete.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import random

# ===== LOAD MODEL FUNCTION =====

def load_trained_model(checkpoint_path, num_classes=2, model_name='swinv2_tiny_window16_256', 
                       device='cuda', min_size=1024, max_size=1024):
    """
    Load trained Swin Mask R-CNN model from checkpoint
    Note: This uses the get_swin_maskrcnn_model function from the training cell
    """
    
    # Create model architecture (reuse function from training cell)
    model = get_swin_maskrcnn_model(
        num_classes=num_classes,
        model_name=model_name,
        pretrained_backbone=False,  # Loading trained weights
        min_size=min_size,
        max_size=max_size
    )
    
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    print(f'✓ Loaded model from epoch {checkpoint.get("epoch", "unknown")}')
    if 'val_loss' in checkpoint:
        print(f'  Validation loss: {checkpoint["val_loss"]:.4f}')
    
    return model


# ===== INFERENCE ON VALIDATION LOADER =====

def compare_prediction_vs_ground_truth(model, val_loader, num_samples=5, 
                                       device='cuda', score_threshold=0.6):
    """
    Compare model predictions with ground truth on validation set
    
    Args:
        model: Trained model
        val_loader: Validation DataLoader
        num_samples: Number of random samples to visualize
        device: Device to run on
        score_threshold: Confidence threshold for predictions
    """
    model.eval()
    
    # Get all indices from validation set
    dataset_size = len(val_loader.dataset)
    random_indices = random.sample(range(dataset_size), min(num_samples, dataset_size))
    
    print(f"Selected {len(random_indices)} random validation samples: {random_indices}")
    
    for sample_idx, idx in enumerate(random_indices, 1):
        print(f"\n{'='*60}")
        print(f"Sample {sample_idx}/{len(random_indices)} - Dataset Index: {idx}")
        print(f"{'='*60}")
        
        # Get ground truth
        image_tensor, target = val_loader.dataset[idx]
        
        # Convert tensor to numpy for visualization
        image_np = image_tensor.permute(1, 2, 0).cpu().numpy()
        
        # Get predictions
        with torch.no_grad():
            model.eval()
            pred = model([image_tensor.to(device)])[0]
        
        # Filter predictions by score threshold
        keep = pred['scores'] > score_threshold
        predictions = {
            'boxes': pred['boxes'][keep].cpu().numpy(),
            'labels': pred['labels'][keep].cpu().numpy(),
            'scores': pred['scores'][keep].cpu().numpy(),
            'masks': pred['masks'][keep].cpu().numpy()
        }
        
        # Print statistics
        num_gt = len(target['boxes'])
        num_pred = len(predictions['boxes'])
        print(f"Ground Truth: {num_gt} instances")
        print(f"Predictions: {num_pred} instances (threshold={score_threshold})")
        
        if num_pred > 0:
            print(f"Prediction scores: {predictions['scores']}")
            print(f"Average confidence: {predictions['scores'].mean():.3f}")
        else:
            print(f"No predictions above threshold {score_threshold}")
        
        # Create comparison visualization
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # ===== ROW 1: GROUND TRUTH =====
        
        # Original image
        axes[0, 0].imshow(image_np)
        axes[0, 0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0, 0].axis('off')
        
        # Ground truth boxes
        axes[0, 1].imshow(image_np)
        for box in target['boxes']:
            x1, y1, x2, y2 = box.cpu().numpy()
            rect = mpatches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor='green', facecolor='none'
            )
            axes[0, 1].add_patch(rect)
        axes[0, 1].set_title(f'Ground Truth Boxes (n={num_gt})', 
                            fontsize=14, fontweight='bold', color='green')
        axes[0, 1].axis('off')
        
        # Ground truth masks
        if len(target['masks']) > 0:
            gt_mask = target['masks'].sum(dim=0).cpu().numpy()
        else:
            gt_mask = np.zeros((image_np.shape[0], image_np.shape[1]))
        axes[0, 2].imshow(gt_mask, cmap='hot')
        axes[0, 2].set_title(f'Ground Truth Masks (n={num_gt})', 
                            fontsize=14, fontweight='bold', color='green')
        axes[0, 2].axis('off')
        
        # ===== ROW 2: PREDICTIONS =====
        
        # Original image
        axes[1, 0].imshow(image_np)
        axes[1, 0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[1, 0].axis('off')
        
        # Predicted boxes
        axes[1, 1].imshow(image_np)
        for box, score in zip(predictions['boxes'], predictions['scores']):
            x1, y1, x2, y2 = box
            rect = mpatches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor='red', facecolor='none'
            )
            axes[1, 1].add_patch(rect)
            # Add confidence score
            axes[1, 1].text(x1, y1-5, f'{score:.2f}',
                           color='white', fontsize=10,
                           bbox=dict(facecolor='red', alpha=0.7))
        axes[1, 1].set_title(f'Predicted Boxes (n={num_pred})', 
                            fontsize=14, fontweight='bold', color='red')
        axes[1, 1].axis('off')
        
        # Predicted masks
        if len(predictions['masks']) > 0:
            pred_mask = predictions['masks'][:, 0, :, :].sum(axis=0)
        else:
            pred_mask = np.zeros_like(gt_mask)
        axes[1, 2].imshow(pred_mask, cmap='hot')
        axes[1, 2].set_title(f'Predicted Masks (n={num_pred})', 
                            fontsize=14, fontweight='bold', color='red')
        axes[1, 2].axis('off')
        
        # Add overall title
        fig.suptitle(f'Validation Sample {sample_idx} - Index {idx}', 
                     fontsize=16, fontweight='bold', y=0.98)
        
        plt.tight_layout()
        
        # Save
        save_path = f'./val_comparison_sample_{sample_idx}_idx_{idx}.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Saved visualization to {save_path}")
        
        plt.show()
        plt.close()


def calculate_iou(box1, box2):
    """
    Calculate IoU (Intersection over Union) between two bounding boxes
    
    Args:
        box1, box2: [x1, y1, x2, y2] format
    
    Returns:
        iou: Float between 0 and 1
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    if x2 < x1 or y2 < y1:
        return 0.0
    
    intersection = (x2 - x1) * (y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0


def calculate_mask_iou(mask1, mask2):
    """
    Calculate IoU between two binary masks
    
    Args:
        mask1, mask2: Binary masks (numpy arrays)
    
    Returns:
        iou: Float between 0 and 1
    """
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    
    return intersection / union if union > 0 else 0


def match_predictions_to_targets(pred_boxes, target_boxes, iou_threshold=0.5):
    """
    Match predicted boxes to target boxes using Hungarian matching
    
    Args:
        pred_boxes: Predicted boxes [N, 4]
        target_boxes: Target boxes [M, 4]
        iou_threshold: Minimum IoU for a match
    
    Returns:
        matches: List of (pred_idx, target_idx, iou) tuples
        unmatched_preds: List of unmatched prediction indices
        unmatched_targets: List of unmatched target indices
    """
    if len(pred_boxes) == 0 or len(target_boxes) == 0:
        return [], list(range(len(pred_boxes))), list(range(len(target_boxes)))
    
    # Compute IoU matrix
    iou_matrix = np.zeros((len(pred_boxes), len(target_boxes)))
    for i, pred_box in enumerate(pred_boxes):
        for j, target_box in enumerate(target_boxes):
            iou_matrix[i, j] = calculate_iou(pred_box, target_box)
    
    # Greedy matching (can be replaced with Hungarian algorithm for better results)
    matches = []
    matched_preds = set()
    matched_targets = set()
    
    # Sort by IoU descending
    iou_flat = []
    for i in range(len(pred_boxes)):
        for j in range(len(target_boxes)):
            if iou_matrix[i, j] >= iou_threshold:
                iou_flat.append((i, j, iou_matrix[i, j]))
    
    iou_flat.sort(key=lambda x: x[2], reverse=True)
    
    for pred_idx, target_idx, iou in iou_flat:
        if pred_idx not in matched_preds and target_idx not in matched_targets:
            matches.append((pred_idx, target_idx, iou))
            matched_preds.add(pred_idx)
            matched_targets.add(target_idx)
    
    unmatched_preds = [i for i in range(len(pred_boxes)) if i not in matched_preds]
    unmatched_targets = [i for i in range(len(target_boxes)) if i not in matched_targets]
    
    return matches, unmatched_preds, unmatched_targets


def evaluate_on_validation_set(model, val_loader, device='cuda', 
                               score_threshold=0.3, iou_threshold=0.5):
    """
    Evaluate model on entire validation set and compute statistics including IoU
    
    Args:
        model: Trained model
        val_loader: Validation DataLoader
        device: Device to run on
        score_threshold: Confidence threshold for predictions
        iou_threshold: IoU threshold for matching predictions to ground truth
    
    Returns:
        stats: Dictionary with evaluation statistics
    """
    model.eval()
    
    total_gt_instances = 0
    total_pred_instances = 0
    total_true_positives = 0
    total_false_positives = 0
    total_false_negatives = 0
    
    all_scores = []
    all_ious = []
    all_box_ious = []
    all_mask_ious = []
    
    dataset_size = len(val_loader.dataset)
    
    print(f"\nEvaluating on ALL {dataset_size} validation samples...")
    
    for idx in range(dataset_size):
        if (idx + 1) % 20 == 0:
            print(f"Processed {idx+1}/{dataset_size} samples...")
        
        # Get data
        image_tensor, target = val_loader.dataset[idx]
        
        # Get predictions
        with torch.no_grad():
            pred = model([image_tensor.to(device)])[0]
        
        # Filter by threshold
        keep = pred['scores'] > score_threshold
        pred_boxes = pred['boxes'][keep].cpu().numpy()
        pred_scores = pred['scores'][keep].cpu().numpy()
        pred_masks = pred['masks'][keep].cpu().numpy()
        
        target_boxes = target['boxes'].cpu().numpy()
        target_masks = target['masks'].cpu().numpy()
        
        # Update counts
        total_gt_instances += len(target_boxes)
        total_pred_instances += len(pred_boxes)
        all_scores.extend(pred_scores.tolist())
        
        # Match predictions to targets
        matches, unmatched_preds, unmatched_targets = match_predictions_to_targets(
            pred_boxes, target_boxes, iou_threshold
        )
        
        # Calculate metrics
        true_positives = len(matches)
        false_positives = len(unmatched_preds)
        false_negatives = len(unmatched_targets)
        
        total_true_positives += true_positives
        total_false_positives += false_positives
        total_false_negatives += false_negatives
        
        # Calculate IoUs for matched pairs
        for pred_idx, target_idx, box_iou in matches:
            all_box_ious.append(box_iou)
            
            # Calculate mask IoU for matched pairs
            pred_mask = (pred_masks[pred_idx, 0] > 0.5).astype(np.uint8)
            # target_masks is already numpy array from .cpu().numpy() above
            target_mask = target_masks[target_idx].astype(np.uint8)
            mask_iou = calculate_mask_iou(pred_mask, target_mask)
            all_mask_ious.append(mask_iou)
            all_ious.append(mask_iou)  # Use mask IoU as overall IoU
    
    # Compute metrics
    precision = total_true_positives / (total_true_positives + total_false_positives) if (total_true_positives + total_false_positives) > 0 else 0
    recall = total_true_positives / (total_true_positives + total_false_negatives) if (total_true_positives + total_false_negatives) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    stats = {
        'num_samples': dataset_size,
        'total_ground_truth': total_gt_instances,
        'total_predictions': total_pred_instances,
        'true_positives': total_true_positives,
        'false_positives': total_false_positives,
        'false_negatives': total_false_negatives,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'avg_gt_per_image': total_gt_instances / dataset_size,
        'avg_pred_per_image': total_pred_instances / dataset_size,
        'avg_confidence': np.mean(all_scores) if all_scores else 0,
        'median_confidence': np.median(all_scores) if all_scores else 0,
        'min_confidence': np.min(all_scores) if all_scores else 0,
        'max_confidence': np.max(all_scores) if all_scores else 0,
        'avg_box_iou': np.mean(all_box_ious) if all_box_ious else 0,
        'avg_mask_iou': np.mean(all_mask_ious) if all_mask_ious else 0,
        'median_box_iou': np.median(all_box_ious) if all_box_ious else 0,
        'median_mask_iou': np.median(all_mask_ious) if all_mask_ious else 0,
        'score_threshold': score_threshold,
        'iou_threshold': iou_threshold
    }
    
    return stats


# ===== MAIN EXECUTION =====

# Configuration
CHECKPOINT_PATH = '/kaggle/working/swin_checkpoints/best_model.pth'
MODEL_NAME = 'swinv2_tiny_window16_256'
NUM_CLASSES = 2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SCORE_THRESHOLD = 0.9  # Very low threshold to see early predictions
NUM_SAMPLES = 5

print(f'{"="*60}')
print('Validation Set Inference - Predictions vs Ground Truth')
print(f'{"="*60}')
print(f'Device: {DEVICE}')
print(f'Model: {MODEL_NAME}')
print(f'Score Threshold: {SCORE_THRESHOLD}')
print(f'Number of Samples: {NUM_SAMPLES}')
print(f'{"="*60}\n')

# Load model
print('Loading trained model...')
model = load_trained_model(
    checkpoint_path=CHECKPOINT_PATH,
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES,
    device=DEVICE,
    min_size=1024,
    max_size=1024
)

# Compare predictions vs ground truth on random samples
print('\n--- Visualizing Random Validation Samples ---')
compare_prediction_vs_ground_truth(
    model=model,
    val_loader=val_loader,
    num_samples=NUM_SAMPLES,
    device=DEVICE,
    score_threshold=SCORE_THRESHOLD
)

# Evaluate on entire validation set (optional)
print(f'\n{"="*60}')
print('--- Full Validation Set Evaluation ---')
print(f'{"="*60}')

stats = evaluate_on_validation_set(
    model=model,
    val_loader=val_loader,
    device=DEVICE,
    score_threshold=SCORE_THRESHOLD,
    iou_threshold=0.5  # IoU threshold for matching predictions to ground truth
)

print(f"\n{'='*60}")
print("VALIDATION RESULTS")
print(f"{'='*60}")
print(f"\n📊 Dataset Statistics:")
print(f"  Total samples evaluated: {stats['num_samples']}")
print(f"  Total ground truth instances: {stats['total_ground_truth']}")
print(f"  Total predicted instances: {stats['total_predictions']}")
print(f"  Avg GT instances per image: {stats['avg_gt_per_image']:.2f}")
print(f"  Avg predictions per image: {stats['avg_pred_per_image']:.2f}")

print(f"\n🎯 Detection Metrics (IoU threshold = {stats['iou_threshold']}):")
print(f"  True Positives (TP): {stats['true_positives']}")
print(f"  False Positives (FP): {stats['false_positives']}")
print(f"  False Negatives (FN): {stats['false_negatives']}")
print(f"  Precision: {stats['precision']:.4f} ({stats['precision']*100:.2f}%)")
print(f"  Recall: {stats['recall']:.4f} ({stats['recall']*100:.2f}%)")
print(f"  F1 Score: {stats['f1_score']:.4f}")

print(f"\n📈 IoU Metrics:")
print(f"  Average Box IoU: {stats['avg_box_iou']:.4f}")
print(f"  Average Mask IoU: {stats['avg_mask_iou']:.4f}")
print(f"  Median Box IoU: {stats['median_box_iou']:.4f}")
print(f"  Median Mask IoU: {stats['median_mask_iou']:.4f}")

if stats['avg_confidence'] > 0:
    print(f"\n💯 Confidence Statistics:")
    print(f"  Average confidence: {stats['avg_confidence']:.3f}")
    print(f"  Median confidence: {stats['median_confidence']:.3f}")
    print(f"  Confidence range: [{stats['min_confidence']:.3f}, {stats['max_confidence']:.3f}]")
else:
    print(f"\n⚠️  No predictions made above threshold {stats['score_threshold']}")

print(f"\n⚙️  Thresholds:")
print(f"  Score threshold: {stats['score_threshold']}")
print(f"  IoU threshold: {stats['iou_threshold']}")

print(f'\n{"="*60}')
print('✓ Validation inference complete!')
print(f'{"="*60}')